In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from data_loader_cropping_padding import CroppedPaddedDataset
from torch.utils.data import random_split, DataLoader
import matplotlib.pyplot as plt
import sys

In [7]:
sys.path.append("../dataloaders")


In [8]:
dataset = CroppedPaddedDataset("../labels.csv", "../images")

Image width 400 and height 200
Found 402 valid samples


In [9]:
class CNNRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, 1)  
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = self.regressor(x)
        return x

In [10]:
#Training and validation data
val_split = 0.2  
n_total = len(dataset)
n_val = int(n_total * val_split)
n_train = n_total - n_val

train_set, val_set = random_split(dataset, [n_train, n_val])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = CNNRegressor().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)


Using device: cpu


In [ ]:
num_epochs = 100
train_mse_list = []
val_mse_list = []
val_mae_list = []

for epoch in range(num_epochs):
    #Training
    model.train()
    running_loss = 0.0
    n = 0

    for imgs, labels, names in train_loader:   
        imgs = imgs.to(device)
        labels = labels.float().view(-1, 1).to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = imgs.size(0)
        running_loss += loss.item() * imgs.size(0)
        n += batch_size

    print(f"Epoch {epoch+1}/{num_epochs} - Train MSE: {running_loss / n:.4f}")

    #Validation
    model.eval()
    val_loss = 0.0
    val_mae = 0.0
    n_val = 0

    with torch.no_grad():
        for imgs, labels, names in val_loader:
            imgs = imgs.to(device)
            labels = labels.float().view(-1, 1).to(device)

            outputs = model(imgs)

            #MSE 
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)

            #MAE 
            val_mae += torch.abs(outputs - labels).sum().item()

            n_val += imgs.size(0)

    train_mse = running_loss / n
    val_mse = val_loss / n_val
    val_mae_epoch = val_mae / n_val

    train_mse_list.append(train_mse)
    val_mse_list.append(val_mse)
    val_mae_list.append(val_mae_epoch)

    print(f"Val MSE: {val_loss / n_val:.4f} | Val MAE: {val_mae / n_val:.4f}")

Epoch 1/100 - Train MSE: 70.5839
Val MSE: 39.5140 | Val MAE: 4.6986
Epoch 2/100 - Train MSE: 68.8348
Val MSE: 37.7311 | Val MAE: 4.5104
Epoch 3/100 - Train MSE: 65.8311
Val MSE: 34.7062 | Val MAE: 4.1767
Epoch 4/100 - Train MSE: 60.1400
Val MSE: 29.9733 | Val MAE: 3.7004
Epoch 5/100 - Train MSE: 53.1890
Val MSE: 25.2175 | Val MAE: 3.3127
Epoch 6/100 - Train MSE: 45.9717
Val MSE: 20.6111 | Val MAE: 3.0057
Epoch 7/100 - Train MSE: 38.4123
Val MSE: 17.1933 | Val MAE: 3.0435
Epoch 8/100 - Train MSE: 32.6292
Val MSE: 16.6928 | Val MAE: 3.3496
Epoch 9/100 - Train MSE: 30.5415
Val MSE: 17.9725 | Val MAE: 3.6292
Epoch 10/100 - Train MSE: 29.9847
Val MSE: 18.0120 | Val MAE: 3.6350
Epoch 11/100 - Train MSE: 29.9800
Val MSE: 18.3804 | Val MAE: 3.6870
Epoch 12/100 - Train MSE: 29.9667
Val MSE: 17.6147 | Val MAE: 3.5710
Epoch 13/100 - Train MSE: 29.8686
Val MSE: 18.2076 | Val MAE: 3.6621
Epoch 14/100 - Train MSE: 29.7335
Val MSE: 17.5916 | Val MAE: 3.5670
Epoch 15/100 - Train MSE: 29.7960
Val MSE: 

In [ ]:
epochs = range(1, num_epochs + 1)

plt.plot(epochs, train_mse_list, label='Train MSE')
plt.plot(epochs, val_mse_list, label='Val MSE')
#plt.plot(epochs, val_mae_list, label='Val MAE')
plt.xlabel('Epoch')
plt.ylabel('Error')
plt.legend()
plt.title('Training and Validation Metrics')
plt.show()